# Experiment 046: GPT-2 XL LoRA SFT to GRPO (Colab)

Fixed 3 seed GPT-2 XL LoRA package. Each seed starts from the same immutable pretrained model and evaluates its SFT and GRPO endpoints only after the technical pilot has selected the card!


## 0. Compute units and Drive


In [ ]:
NOTEBOOK_NAME = '046_colab_gpt2_b15_lora_sft_grpo'
import glob, json, os, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 99
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)
old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as handle:
            old = json.load(handle)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as handle:
            handle.write(json.dumps(old) + '\n')
        os.remove(old_path)
    if not os.path.exists(open_path):
        with open(open_path, 'w') as handle:
            json.dump({'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                       'notebook': NOTEBOOK_NAME, 'colab_cu_before': cu_before,
                       'unit_rate': unit_rate}, handle, indent=2)


session 20260827_082327 | cu_before = 99
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Repository and dependencies


In [ ]:
import base64
import getpass
import os
from pathlib import Path
import subprocess
import sys

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [ ]:
import os, subprocess, sys, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
scratch_package = os.path.join(repo_root, '002_GPT2_From_Scratch')
if not os.path.isdir(scratch_package):
    scratch_package = os.path.join(repo_root, '003_My GPT From Scratch')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', scratch_package], check=True)
sys.path.insert(0, scratch_package)

import gc
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from gpt2_scratch.weights import REFERENCE_ALIAS, REFERENCE_REVISION

GPT2TokenizerFast.from_pretrained(
    REFERENCE_ALIAS,
    revision=REFERENCE_REVISION,
)
reference = GPT2LMHeadModel.from_pretrained(
    REFERENCE_ALIAS,
    revision=REFERENCE_REVISION,
    dtype=torch.float32,
)
del reference
gc.collect()

print('dependencies installed')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

dependencies installed


In [ ]:
import getpass, os
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None


## 2. Registered setup


In [ ]:
import json, os, shutil, subprocess, sys, time, torch
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s12_eval import IntegrityError
from scripts.s46_gpt2_xl import (ALLOCATOR_CONFIG, BASE_TAG, CONFIGS_046, SEEDS_046,
                                 a1_state, arm_state,
                                 arm_tag, check_card_lock, check_card_vram,
                                 check_registered_config,
                                 commit_a1_conclusion, commit_arm_complete,
                                 config_path, eval_state, training_record,
                                 verify_technical_records)

assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == ALLOCATOR_CONFIG
assert cu_before is not None and unit_rate is not None, 'fill the compute fields first'
cfgs = {seed: check_registered_config(load_config(config_path(seed))) for seed in SEEDS_046}
lock = check_card_lock(cfgs[0])
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert 'A100' in gpu, 'need an A100 runtime'
check_card_vram(lock, total_gib)
assert shutil.disk_usage(ROOT).free / 1024**3 >= 40, 'need at least 40 GiB free on Drive'
#SEEDS_TO_RUN = (0,)
SEEDS_TO_RUN = (2,)
# select the fixed seeds assigned to this session
assert set(SEEDS_TO_RUN) <= set(SEEDS_046)

BASE_RESULT = os.path.join(RESULTS, BASE_TAG)
BASE_DIR = os.path.join(ROOT, 'checkpoints', BASE_TAG)
BASE_RECORD = os.path.join(BASE_RESULT, '046_base_manifest.json')
TECH = os.path.join(RESULTS, '046_technical')
verify_technical_records(
    cfgs[0], BASE_RECORD, os.path.join(TECH, '046_xl_parity.json'),
    os.path.join(TECH, '046_token_audit.json'))


{'base': {'completed': True,
  'experiment': '046',
  'role': 'immutable_base',
  'artifact': '/content/drive/MyDrive/inm363/checkpoints/046_COLAB_FULL_GPT2_B15_base',
  'artifact_sha256': {'artifact_type.json': '7c70301d0243f84d115d9e76cc66a4aa9fde74cdc3ecfe08091802a65d1c93c9',
   'config.json': '48bd45e1cfdeeb968dc7387dbc9b0bca472449969029aa4f21a9dcb43ed4099a',
   'generation_config.json': '6c53044d46226d4ceb2410ad49fe751b37a16cde6c518679b652dfa22d5970a1',
   'model.safetensors': 'df636f168a87529b6bfa55c9555682c95a23a0d0b41eed0cbbfc85b42a480bd5',
   'model_base.json': '5f742459ed1748c36e982ad5058cb1ec8cf5172073252afb51baf8852f0cdf56',
   'tokenizer.json': '1fe93b6152957cf9cfd6d89002467f789ce8b3f3e000b3a2edf27c808ddd0b9e',
   'tokenizer_config.json': 'caab7b8a5a261a1d4cb142c2488acd9cb1c5c0154f681e5a38614a3a2c0cf8ce'},
  'source': 'gpt2-xl',
  'source_revision': '15ea56dee5df4983c59b2538573817e1667135e2'},
 'parity': {'completed': True,
  'experiment': '046',
  'source': 'openai-commun

## 3. Wiring checks


In [ ]:
checks = subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_scratch_gpt_pipeline.py', 'tests/test_042_m124.py',
    'tests/test_046_gpt2_xl.py', 'tests/test_046_notebooks.py',
    '-k', 'not test_046_notebooks_are_clean_and_unexecuted and not test_046_units_are_blank',
], capture_output=True, text=True)
print(checks.stdout)
if checks.stderr:
    print(checks.stderr)
checks.check_returncode()


..................................................                       [100%]
50 passed, 2 deselected in 25.54s



## 4. Worker helpers


In [ ]:
def active_workers(cfg_path):
    rows = subprocess.run(['ps', '-eo', 'pid=,args='], capture_output=True,
                          text=True, check=True).stdout
    return [row.strip() for row in rows.splitlines()
            if 'scripts/s46_worker.py' in row and f'--config {cfg_path}' in row]

def run_worker(command, status_path, result_dir, label, test_passes=0):
    if active_workers(command[command.index('--config') + 1]):
        raise RuntimeError(f'{label}: another worker is active')
    if os.path.exists(status_path):
        old = read_json(status_path)
        if old.get('error_type') not in (None, 'KeyboardInterrupt') and old.get('completed') is not True:
            raise RuntimeError(f'{label}: previous worker failure needs review')
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus'}
    status['returncode'] = process.returncode
    status['label'] = label
    status['mmlu_test_passes'] = test_passes
    with open(os.path.join(result_dir, '046_attempts.jsonl'), 'a') as handle:
        handle.write(json.dumps(status) + '\n')
    with open(os.path.join(RESULTS, 'compute_ledger.jsonl'), 'a') as handle:
        handle.write(json.dumps({
            'run_tag': os.path.basename(result_dir), 'condition': label,
            'timings_sec': {label: round(time.time() - started, 1)},
            'train_steps': status.get('completed_updates'),
            'completed': status.get('completed') is True,
            'gpu': status.get('gpu'), 'mmlu_test_passes': test_passes,
        }) + '\n')
    return status

def command(task, cfg_path, status_path, result_dir, **kwargs):
    values = [sys.executable, '-u', 'scripts/s46_worker.py', '--task', task,
              '--config', cfg_path, '--status', status_path,
              '--result-dir', result_dir]
    for key, value in kwargs.items():
        values.extend(['--' + key.replace('_', '-'), value])
    return values

def fresh_scratch(label):
    path = f'/content/{label}_{session_id}'
    shutil.rmtree(path, ignore_errors=True)
    return path


## 5. Shared base endpoint


In [ ]:
base_binding = read_json(BASE_RECORD)
if eval_state(BASE_RESULT, 'base', base_binding) == 'absent':
    scratch = fresh_scratch('046_base_eval')
    status_path = os.path.join(BASE_RESULT, 'base_eval_status.json')
    status = run_worker(command(
        'eval', config_path(0), status_path, BASE_RESULT,
        branch='base', model_path=BASE_DIR, scratch_dir=scratch,
        record=BASE_RECORD), status_path, BASE_RESULT, '046_base_eval', test_passes=2)
    if status.get('completed') is not True:
        raise RuntimeError(f'base evaluation failed: {status}')
    shutil.rmtree(scratch, ignore_errors=True)
elif eval_state(BASE_RESULT, 'base', base_binding) != 'committed':
    raise IntegrityError('base endpoint changed')


## 6. Fixed seed pipelines


In [ ]:
def run_seed(seed):
    cfg = cfgs[seed]
    cfg_path = config_path(seed)
    tag = arm_tag(seed)
    result_dir = os.path.join(RESULTS, tag)
    checkpoint_dir = os.path.join(ROOT, 'checkpoints', tag)
    sft_dir = os.path.join(checkpoint_dir, 'sft')
    merged_dir = os.path.join(checkpoint_dir, 'sft_merged')
    grpo_dir = os.path.join(checkpoint_dir, 'grpo')
    os.makedirs(result_dir, exist_ok=True)

    state = arm_state(result_dir)
    if state['state'] in ('committed', 'a1'):
        return state['state']
    if state['state'] == 'pending':
        commit_a1_conclusion(result_dir)
        return 'a1'
    if state['state'] == 'failed':
        raise RuntimeError(f'{tag}: previous GRPO failure needs review')

    sft_status = os.path.join(result_dir, 'sft_worker_status.json')
    if not os.path.exists(os.path.join(result_dir, '046_sft_training.json')):
        status = run_worker(command(
            'sft', cfg_path, sft_status, result_dir,
            base_record=BASE_RECORD, out_dir=sft_dir),
            sft_status, result_dir, f'{tag}_sft')
        if status.get('completed') is not True:
            raise RuntimeError(f'{tag}: SFT failed: {status}')
    sft_record = training_record(result_dir, 'sft')

    merge_status = os.path.join(result_dir, 'merge_worker_status.json')
    status = run_worker(command(
        'merge', cfg_path, merge_status, result_dir,
        adapter_dir=sft_dir, start_dir=merged_dir, out_dir=grpo_dir),
        merge_status, result_dir, f'{tag}_merge')
    if status.get('completed') is not True:
        raise RuntimeError(f'{tag}: merge failed: {status}')

    grpo_status = os.path.join(result_dir, 'grpo_worker_status.json')
    if not os.path.exists(os.path.join(result_dir, '046_grpo_training.json')):
        status = run_worker(command(
            'grpo', cfg_path, grpo_status, result_dir,
            sft_record=os.path.join(result_dir, '046_sft_training.json'),
            start_dir=merged_dir, out_dir=grpo_dir,
            precision_report=os.path.join(result_dir, 'precision_report.json')),
            grpo_status, result_dir, f'{tag}_grpo')
        if status.get('completed') is not True:
            state, _ = a1_state(result_dir)
            if state == 'pending':
                commit_a1_conclusion(result_dir)
                return 'a1'
            raise RuntimeError(f'{tag}: GRPO failed: {status}')
    grpo_record = training_record(result_dir, 'grpo')

    for branch, binding, adapter in (
            ('sft', sft_record, sft_dir), ('grpo', grpo_record, grpo_dir)):
        state = eval_state(result_dir, branch, binding)
        if state == 'tampered':
            raise IntegrityError(f'{tag}: {branch} endpoint changed')
        if state == 'absent':
            scratch = fresh_scratch(f'046_{seed}_{branch}_eval')
            status_path = os.path.join(result_dir, f'{branch}_eval_status.json')
            status = run_worker(command(
                'eval', cfg_path, status_path, result_dir,
                branch=branch, adapter_dir=adapter, scratch_dir=scratch,
                record=os.path.join(result_dir, f'046_{branch}_training.json')),
                status_path, result_dir, f'{tag}_{branch}_eval', test_passes=2)
            if status.get('completed') is not True:
                raise RuntimeError(f'{tag}: {branch} evaluation failed: {status}')
            shutil.rmtree(scratch, ignore_errors=True)
    commit_arm_complete(result_dir)
    return 'committed'

outcomes = {seed: run_seed(seed) for seed in SEEDS_TO_RUN}
print(outcomes)


Streaming output truncated to the last 5000 lines.
│ │ Kate's        │ A and B:      │               │              │           │ │
│ │ A.            │ 2/2...2/2     │               │              │           │ │
│ │ egocentrism.  │ He's either   │               │              │           │ │
│ │ B. lack of    │ out ratio is  │               │              │           │ │
│ │ object        │ pinned at     │               │              │           │ │
│ │ permanence.   │ about a 1:2   │               │              │           │ │
│ │ C. inability  │ ratio of 3    │               │              │           │ │
│ │ to conserve.  │ 2/3 - 2/3,    │               │              │           │ │
│ │ D. animism.   │ and the odds  │               │              │           │ │
│ │ <think>       │ that he will  │               │              │           │ │
│ │               │ error is      │               │              │           │ │
│ │               │ pinned at     │               │       

## 7. Close compute units


In [ ]:
cu_after = 29.33
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif os.path.exists(open_path):
    with open(open_path) as handle:
        record = json.load(handle)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as handle:
        handle.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


cu_after = FILL ME
fill cu_after, then rerun this cell
